# 04 — Hooks Demo

This notebook demonstrates how Claude Code hooks work by simulating
pre-tool-use and post-tool-use hook behavior in Python.

## Learning Objectives
1. Understand the hook execution model (stdin JSON → exit code)
2. Implement a pre-tool-use hook that blocks sensitive file access
3. Implement a post-tool-use hook that runs formatting after edits

## Setup

In [ ]:
import json
from dataclasses import dataclass
from typing import Any

print("Ready.")

## Hook Data Model

Hooks receive tool call data as JSON via stdin. Let's model that.

In [ ]:
@dataclass
class ToolCall:
    """Represents a tool call that hooks can inspect."""
    session_id: str
    tool_name: str
    tool_input: dict[str, Any]

    def to_json(self) -> str:
        return json.dumps({
            "session_id": self.session_id,
            "tool_name": self.tool_name,
            "tool_input": self.tool_input,
        }, indent=2)


@dataclass
class HookResult:
    """Result of a hook execution."""
    exit_code: int  # 0 = allow, 2 = block
    message: str = ""

    @property
    def allowed(self) -> bool:
        return self.exit_code == 0


# Example tool call
tc = ToolCall(
    session_id="sess_abc123",
    tool_name="read",
    tool_input={"file_path": "/project/.env"},
)
print("Example tool call JSON:")
print(tc.to_json())

## Pre-Tool-Use Hook: Block `.env` Access

In [ ]:
BLOCKED_PATTERNS = [".env", ".secret", "credentials", "private_key"]


def pre_hook_block_sensitive(tool_call: ToolCall) -> HookResult:
    """Pre-tool-use hook that blocks access to sensitive files."""
    file_path = (
        tool_call.tool_input.get("file_path", "")
        or tool_call.tool_input.get("path", "")
    )

    for pattern in BLOCKED_PATTERNS:
        if pattern in file_path.lower():
            return HookResult(
                exit_code=2,
                message=f'BLOCKED: Access to "{file_path}" denied. '
                        f'Matches sensitive pattern: {pattern}',
            )

    return HookResult(exit_code=0, message="Allowed.")


# Test cases
test_cases = [
    ToolCall("s1", "read", {"file_path": "/project/.env"}),
    ToolCall("s1", "read", {"file_path": "/project/src/app.py"}),
    ToolCall("s1", "grep", {"path": "/project/.env.local", "pattern": "KEY"}),
    ToolCall("s1", "read", {"file_path": "/project/credentials.json"}),
    ToolCall("s1", "read", {"file_path": "/project/README.md"}),
]

print("Pre-hook results:")
for tc in test_cases:
    result = pre_hook_block_sensitive(tc)
    status = "✅ ALLOW" if result.allowed else "🚫 BLOCK"
    path = tc.tool_input.get("file_path", tc.tool_input.get("path", ""))
    print(f"  {status} | {tc.tool_name}({path}) → {result.message}")

## Post-Tool-Use Hook: Auto-Format After Edit

In [ ]:
def post_hook_format_check(tool_call: ToolCall) -> HookResult:
    """Post-tool-use hook that checks if formatting is needed."""
    file_path = tool_call.tool_input.get("file_path", "")

    if file_path.endswith(".py"):
        return HookResult(
            exit_code=0,
            message=f"Run: black {file_path}",
        )
    elif file_path.endswith((".ts", ".tsx", ".js", ".jsx")):
        return HookResult(
            exit_code=0,
            message=f"Run: npx prettier --write {file_path}",
        )
    else:
        return HookResult(exit_code=0, message="No formatter configured.")


# Test cases
edit_calls = [
    ToolCall("s1", "write", {"file_path": "src/app.py", "content": "..."}),
    ToolCall("s1", "edit", {"file_path": "src/utils.ts"}),
    ToolCall("s1", "write", {"file_path": "README.md", "content": "..."}),
]

print("Post-hook results:")
for tc in edit_calls:
    result = post_hook_format_check(tc)
    print(f"  {tc.tool_name}({tc.tool_input.get('file_path')}) → {result.message}")

## Simulating the Full Hook Pipeline

In [ ]:
from typing import Callable

HookFn = Callable[[ToolCall], HookResult]


class HookPipeline:
    """Simulates the Claude Code hook execution pipeline."""

    def __init__(self) -> None:
        self.pre_hooks: list[tuple[str, HookFn]] = []
        self.post_hooks: list[tuple[str, HookFn]] = []

    def add_pre_hook(self, matcher: str, hook: HookFn) -> None:
        self.pre_hooks.append((matcher, hook))

    def add_post_hook(self, matcher: str, hook: HookFn) -> None:
        self.post_hooks.append((matcher, hook))

    def _matches(self, matcher: str, tool_name: str) -> bool:
        return any(m.strip() == tool_name for m in matcher.split("|"))

    def execute(self, tool_call: ToolCall, tool_fn: Callable) -> str:
        """Run the full pipeline: pre-hooks → tool → post-hooks."""
        # Pre-hooks
        for matcher, hook in self.pre_hooks:
            if self._matches(matcher, tool_call.tool_name):
                result = hook(tool_call)
                if not result.allowed:
                    print(f"  🚫 Pre-hook blocked: {result.message}")
                    return f"BLOCKED: {result.message}"

        # Execute tool
        print(f"  ⚡ Executing: {tool_call.tool_name}")
        tool_result = tool_fn(tool_call.tool_name, tool_call.tool_input)

        # Post-hooks
        for matcher, hook in self.post_hooks:
            if self._matches(matcher, tool_call.tool_name):
                result = hook(tool_call)
                if result.message:
                    print(f"  📝 Post-hook feedback: {result.message}")

        return tool_result


# Build pipeline
pipeline = HookPipeline()
pipeline.add_pre_hook("read|grep", pre_hook_block_sensitive)
pipeline.add_post_hook("write|edit", post_hook_format_check)


def mock_tool(name: str, params: dict) -> str:
    return f"Tool {name} executed with {params}"


# Test the pipeline
print("=== Pipeline Tests ===")

print("\n1. Read .env (should be blocked):")
pipeline.execute(ToolCall("s1", "read", {"file_path": ".env"}), mock_tool)

print("\n2. Read app.py (should be allowed):")
pipeline.execute(ToolCall("s1", "read", {"file_path": "app.py"}), mock_tool)

print("\n3. Write utils.py (should trigger post-hook):")
pipeline.execute(ToolCall("s1", "write", {"file_path": "utils.py", "content": "..."}), mock_tool)

## The Actual Hook Configuration File

Here's what a real `.claude/settings.local.json` looks like:

In [ ]:
settings = {
    "hooks": {
        "PreToolUse": [
            {
                "matcher": "read|grep",
                "hooks": [
                    {
                        "type": "command",
                        "command": "node ./hooks/read_hook.js",
                    }
                ],
            }
        ],
        "PostToolUse": [
            {
                "matcher": "write|edit",
                "hooks": [
                    {
                        "type": "command",
                        "command": "npx prettier --write $FILE_PATH",
                    }
                ],
            }
        ],
    }
}

print(json.dumps(settings, indent=2))

## Exercise

1. Add a pre-hook that blocks `bash` tool calls containing `rm` or `sudo`
2. Add a post-hook that counts lines of code after a file write
3. Extend the `HookPipeline` to log all hook executions to a list for auditing